# KMDS Notebook Parity: Ingest + Summarize + Save

This notebook demonstrates **notebook parity** for the KMDS ingest workflow using the same backend used by the UI:

1. Ingest a documentation source URL
2. Query using the template `<concept> in <source>`
3. Summarize matched context
4. Save an **exploratory observation** to a KMDS project

Source URL used in this example:
- https://archive.ics.uci.edu/dataset/498/incident+management+process+enriched+event+log

In [ ]:
from pathlib import Path
from pprint import pprint

from kmds.dashboard.service import (
    ingest_documents,
    load_project_snapshot,
    query_indexed_documents,
    update_project_observation,
    delete_project_observation,
    )
from kmds.dashboard.tagger import TaggerClass

In [ ]:
PROJECT_FILE = Path('examples_of_use/analytics/incident_management_ingest_parity.xml').resolve()
SOURCE_URL = 'https://archive.ics.uci.edu/dataset/498/incident+management+process+enriched+event+log'
INGEST_GOAL = (
    'Capture attribute definitions, field semantics, and data quality context '
    'for incident management analytics documentation.'
)
QUERY = 'attribute summary in incident management process enriched event log'
WORKFLOW_NAME = 'incident_management_analytics'

print('Project file:', PROJECT_FILE)
print('Source URL:', SOURCE_URL)
print('Ingest goal:', INGEST_GOAL)
print('Query:', QUERY)

In [ ]:
ingest_result = ingest_documents(
    project_file=str(PROJECT_FILE),
    goal=INGEST_GOAL,
    urls=[SOURCE_URL],
)

print('Indexed fragments:', ingest_result.get('indexed_fragments'))
print('Stored documents:', len(ingest_result.get('stored_documents', [])))
pprint(ingest_result.get('stored_documents', []))

In [ ]:
query_result = query_indexed_documents(
    project_file=str(PROJECT_FILE),
    query=QUERY,
    limit=20,
    comprehensive=True,
)

matches = query_result.get('matches', [])
print('Matches:', len(matches))
print('Structured query:', query_result.get('structured_query'))

for i, match in enumerate(matches[:5], start=1):
    print(f'--- Match {i} ---')
    print('Source:', match.get('source'))
    print('Concept overlap:', match.get('concept_overlap'))
    print('Source overlap:', match.get('source_overlap'))
    print('Finding:', match.get('finding'))
    print()

In [ ]:
tagger = TaggerClass()

if not matches:
    raise ValueError('No matches found. Refine ingest goal/query and re-run.')

summary_text = tagger.summarize_match_context(
    query=QUERY,
    hits=matches,
    max_sentences=6,
)

print('Natural-language summary:')
print(summary_text)

In [ ]:
# Build a validation-friendly exploratory observation and append the generated summary.
observation_text = (
    'On 2026-05-06, exploratory review of incident management documentation identified '
    'attribute-summary context for incident_state, reassignment_count, and opened_time. '
    + summary_text
)

draft = tagger.create_draft(
    goal=QUERY,
    hits=matches,
    phase='Exploratory',
    summary_text=observation_text,
)

print('Draft observation:')
print(draft['observation_text'])
print('Suggested phase:', draft.get('suggested_phase'))
print('Mapping preview:')
pprint(draft.get('mapping_preview', {}))

In [ ]:
save_result = tagger.save_observation(
    text=draft['observation_text'],
    phase='Exploratory',
    workflow_name=WORKFLOW_NAME,
    project_file=str(PROJECT_FILE),
    workflow_type='application',
    create_project=not PROJECT_FILE.exists(),
)

print('Saved to:', save_result.get('project_file'))
print('Logged intent:', save_result.get('mapping', {}).get('intent'))
print('Logged observation type:', save_result.get('mapping', {}).get('observation_type'))

In [ ]:
snapshot = load_project_snapshot(str(PROJECT_FILE))
print('Workflow:', snapshot.get('workflow_name'))
print('Total observations:', snapshot.get('total_observations'))

rows = snapshot.get('observation_rows', [])
latest = rows[-1] if rows else {}
print('Latest observation tag:', latest.get('tag'))
print('Latest observation text:', latest.get('observation'))
print('Latest intent/rationale:', latest.get('rationale'))

## Search API + Edit Observation Example

This step shows how to find an observation using the snapshot search filter and then edit it in the KB.

Search term used: `incident_state` (replace with your own natural-language phrase).

In [ ]:
search_term = 'incident_state'
search_snapshot = load_project_snapshot(
    str(PROJECT_FILE),
    search_text=search_term,
    )

search_rows = search_snapshot.get('observation_rows', [])
print('Search term:', search_term)
print('Search matches:', len(search_rows))

if not search_rows:
    raise ValueError('No observations matched the search term; adjust search_term and rerun.')

target_row = search_rows[0]
print('Target row id:', target_row.get('id'))
print('Target tag:', target_row.get('tag'))
print('Target observation:', target_row.get('observation'))

updated_text = (
    str(target_row.get('observation', '')).strip()
    + ' Updated through notebook search-and-edit API example.'
)

update_result = update_project_observation(
    project_file=str(PROJECT_FILE),
    finding_sequence=int(target_row.get('id') or 0),
    tag=str(target_row.get('tag', '')),
    new_observation_text=updated_text,
    new_intent=str(target_row.get('rationale', '') or 'exploratory'),
    )

print('Update result:')
pprint(update_result)

post_update_snapshot = load_project_snapshot(
    str(PROJECT_FILE),
    search_text='Updated through notebook search-and-edit API example.',
    )

post_rows = post_update_snapshot.get('observation_rows', [])
print('Post-update search matches:', len(post_rows))
assert post_rows, 'Updated observation was not found in KB after edit.'
print('Edit verification passed: updated observation persisted in KB.')

## Notes

- If URL fetch times out, retry or download content and ingest it as a file upload input.
- You can iterate by changing only `QUERY` while keeping the same `INGEST_GOAL` and indexed source set.
- Keep `Ingest goal` broad; use `Query` for immediate retrieval intent.

## Optional Fallback: Local File Ingest Instead Of URL

Use this path when URL fetch is slow or unavailable. The logic remains the same: ingest -> query -> summarize -> save.

1. Download source content locally (for example, HTML/text export).
2. Ingest the file bytes via `uploads`.
3. Re-run the same `<concept> in <source>` query.

In [ ]:
# Optional fallback cell: ingest from a local text/html file instead of URL.
# Keep this disabled unless you have a local source file path to test with.

LOCAL_SOURCE_FILE = Path('examples_of_use/analytics/uci_incident_source.txt').resolve()

if LOCAL_SOURCE_FILE.exists():
    local_ingest_result = ingest_documents(
        project_file=str(PROJECT_FILE),
        goal=INGEST_GOAL,
        uploads=[(LOCAL_SOURCE_FILE.name, LOCAL_SOURCE_FILE.read_bytes())],
    )
    print('Local fallback indexed fragments:', local_ingest_result.get('indexed_fragments'))
else:
    print('Local fallback skipped. Create', LOCAL_SOURCE_FILE, 'to run this path.')

In [ ]:
# Verification cell: confirm the observation text persisted in the KB snapshot.
verify_snapshot = load_project_snapshot(str(PROJECT_FILE))
verify_rows = verify_snapshot.get('observation_rows', [])

matching_rows = [
    row for row in verify_rows
    if 'exploratory review of incident management documentation identified' in str(row.get('observation', '')).lower()
]

print('Verification matches found:', len(matching_rows))
assert matching_rows, 'Expected persisted exploratory attribute summary observation was not found in KB.'
print('Verification passed: observation is persisted in the KB.')